# 00 — Data Acquisition: Tomato Leaf Disease Datasets (Kaggle)

**Dự án:** Smart Greenhouse AI — cà chua
**Notebook này thiết kế để chạy trên Kaggle, không chạy local.**

## Trước khi chạy
1. Panel bên phải: **Settings → Internet → ON** (bắt buộc để gọi Roboflow API và tải Zenodo).
2. **Add-ons → Secrets** → thêm secret tên đúng `ROBOFLOW_API_KEY` với giá trị là API key Roboflow của bạn, rồi **Attach** secret đó vào notebook này (checkbox bật lên) — nếu không attach, cell lấy secret sẽ báo lỗi rõ ràng.
3. Accelerator để **None** — notebook này chỉ tải/giải nén dữ liệu, không train.
4. Chạy xong, bấm **Save Version → Save & Run All (Commit)** để output ở `/kaggle/working/ai/datasets/` được lưu thành Kaggle Dataset, dùng làm input cho `01_build_tomato_leaf_disease_v1.ipynb`.

## Nguồn dữ liệu (nhánh bệnh lá — `tomato_leaf_disease_v1`)
Theo `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md` mục 2:

| # | Dataset | Cách tải | Vai trò |
|---|---|---|---|
| 1 | Tomato Leaf Disease (Roboflow, `universitas-atma-jaya/tomato-leaf-disease-rxcft`) | Roboflow API (cần API key) | **Nguồn chính** — ~8.439 ảnh, 11 class công khai, CC BY 4.0 |
| 2 | Tomato Leaf Disease Detection (Zenodo, record 20004230) | curl trực tiếp (zip YOLOv8, đã export sẵn 640×640 + augmentation) | **Chỉ tải để audit** — theo tài liệu, **chưa gộp ngay** vào `tomato_leaf_disease_v1` cho tới khi xác nhận license/chất lượng/trùng lặp với Roboflow |

4 class mục tiêu (MVP):
```yaml
0: leaf_early_blight
1: leaf_late_blight
2: leaf_mold
3: leaf_septoria_spot
```
Class `Healthy` của Roboflow **không** trở thành 1 class riêng — ảnh lá khỏe được giữ làm **negative sample** (ảnh có, label rỗng) ở notebook build kế tiếp.

## Việc notebook này làm
1. Kết nối Roboflow bằng API key trong Kaggle Secrets, **liệt kê thật các version đang có** của project (không đoán số version) rồi tải version phù hợp nhất về `raw/leaf_roboflow/`.
2. Tải + giải nén Zenodo zip vào `raw/leaf_zenodo/` (giữ nguyên, chỉ để audit ở notebook sau).
3. Ghi `manifests/sources.csv` riêng cho notebook này (2 dòng `leaf_roboflow`/`leaf_zenodo`) — độc lập với `sources.csv` của nhánh độ chín vì mỗi notebook chạy trong session `/kaggle/working/` riêng; notebook build sẽ đọc cả hai qua Add Input nếu cần.
4. Kiểm kê sơ bộ (đếm file ảnh/label) từng nguồn → `manifests/dataset_inventory_leaf.csv`.

**Chưa** làm ở bước này: remap class, gộp Healthy thành negative, audit trực quan, chia split — thuộc `01_build_tomato_leaf_disease_v1.ipynb`.

In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "roboflow"], check=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 87.7 MB/s eta 0:00:00


CompletedProcess(args=['pip', 'install', '-q', 'roboflow'], returncode=0)

In [ ]:
import hashlib
import shutil
import subprocess
import time
from pathlib import Path

import pandas as pd

BASE = Path("/kaggle/working/ai/datasets")
RAW = BASE / "raw"
MANIFESTS = BASE / "manifests"

for d in [RAW, MANIFESTS, RAW / "_archives"]:
    d.mkdir(parents=True, exist_ok=True)

DOWNLOAD_DATE = "2026-08-05"

print("BASE:", BASE)


def download_file(urls, dest_path, retries_per_url=2, timeout=180):
    """urls: 1 URL (str) hoặc list các URL ứng viên, thử lần lượt tới khi thành công."""
    if isinstance(urls, str):
        urls = [urls]
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f"[skip] {dest_path.name} đã tồn tại ({dest_path.stat().st_size / 1e6:.1f} MB)")
        return dest_path

    last_stderr = ""
    for url in urls:
        for attempt in range(1, retries_per_url + 1):
            print(f"[download] {url} -> {dest_path} (lần {attempt})")
            try:
                result = subprocess.run(
                    ["curl", "-L", "--fail", "--retry", "2", "-o", str(dest_path), url],
                    capture_output=True, text=True, timeout=timeout,
                )
            except subprocess.TimeoutExpired:
                # File dở dang do bị kill giữa chừng -> xóa ngay, nếu không lần chạy sau sẽ tưởng
                # nhầm là "đã tải xong" vì dest_path.stat().st_size > 0.
                if dest_path.exists():
                    dest_path.unlink()
                last_stderr = f"timeout sau {timeout}s (đã xóa file dở dang)"
                print(f"[timeout] {last_stderr}")
                time.sleep(3)
                continue
            if result.returncode == 0 and dest_path.exists() and dest_path.stat().st_size > 0:
                print(f"[ok] {dest_path.name} ({dest_path.stat().st_size / 1e6:.1f} MB)")
                return dest_path
            if dest_path.exists():
                dest_path.unlink()
            last_stderr = result.stderr[-500:]
            print(f"[fail] returncode={result.returncode} stderr={last_stderr}")
            time.sleep(3)
        print(f"[next] Chuyển sang URL ứng viên kế tiếp (nếu có) sau khi '{url}' thất bại.")

    raise RuntimeError(f"Không tải được từ bất kỳ URL nào trong {urls}. Lỗi cuối: {last_stderr}")


def sha256_file(path, chunk_size=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def extract_archive(archive_path, dest_dir):
    archive_path = Path(archive_path)
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    suffix = archive_path.suffix.lower()
    if suffix == ".zip":
        shutil.unpack_archive(str(archive_path), str(dest_dir), format="zip")
    else:
        raise ValueError(f"Định dạng chưa hỗ trợ: {suffix}")
    print(f"[extract] {archive_path.name} -> {dest_dir}")
    return dest_dir

### Bước 1 — Kết nối Roboflow + liệt kê version thật (không đoán số version)
Project công khai có thể có nhiều version (bản gốc, bản đã augment...). In hết ra để chọn version **ít/không augmentation** theo đúng khuyến nghị trong tài liệu, thay vì đoán bừa một số version.

In [3]:
from kaggle_secrets import UserSecretsClient

try:
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception as e:
    raise RuntimeError(
        "Không lấy được secret 'ROBOFLOW_API_KEY'. Kiểm tra: (1) đã tạo secret đúng tên này ở "
        "Add-ons -> Secrets, (2) đã bật (attach) secret đó cho notebook hiện tại chưa."
    ) from e

import roboflow

rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)

ROBOFLOW_WORKSPACE = "universitas-atma-jaya"
ROBOFLOW_PROJECT = "tomato-leaf-disease-rxcft"

project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
versions = project.versions()

print(f"Project {ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} có {len(versions)} version:\n")


def version_number_of(v):
    """Roboflow SDK trả .version dạng int hoặc .id dạng 'workspace/project/N' tùy phiên bản package -> thử cả hai."""
    val = getattr(v, "version", None)
    if isinstance(val, int):
        return val
    if isinstance(val, str) and val.isdigit():
        return int(val)
    vid = getattr(v, "id", "")
    tail = str(vid).rstrip("/").split("/")[-1]
    return int(tail) if tail.isdigit() else None


version_info = []
for v in versions:
    n = version_number_of(v)
    n_images = getattr(v, "images", None)
    print(f"- version={n} | id={getattr(v, 'id', '?')} | images={n_images} | "
          f"raw_repr={v}")
    version_info.append({"version": n, "images": n_images})

print("\nGợi ý: chọn version có 'images' gần nhất với ~8.439 (bản gốc, ít augmentation nhất).")

loading Roboflow workspace...
loading Roboflow project...
Project universitas-atma-jaya/tomato-leaf-disease-rxcft có 2 version:

- version=6 | id=universitas-atma-jaya/tomato-leaf-disease-rxcft/6 | images=41366 | raw_repr={
  "name": "Tomato Leaf Disease",
  "type": null,
  "version": "6",
  "augmentation": {
    "rotate": {
      "degrees": 15
    },
    "image": {
      "versions": 3
    },
    "exposure": {
      "percent": 5
    },
    "shear": {
      "horizontal": 15,
      "vertical": 15
    },
    "ninety": {
      "clockwise": true,
      "counter-clockwise": true,
      "upside-down": true
    },
    "flip": {
      "horizontal": true,
      "vertical": true
    }
  },
  "created": 1680332721.638,
  "preprocessing": {
    "auto-orient": true,
    "resize": {
      "width": "512",
      "format": "Stretch to",
      "height": "512"
    },
    "remap": {
      "labels": {
        "Iron Deficiency": {
          "omit": true
        }
      }
    }
  },
  "splits": {
    "valid":

### Bước 2 — Tải version đã chọn về `raw/leaf_roboflow/`
Kết quả Bước 1 cho thấy project này chỉ có 2 version công khai và **cả hai đều đã augmentation** (không có bản gốc thuần). Mặc định tự chọn version có số ảnh (`images`) **nhỏ nhất** trong 2 lựa chọn (ít bản sao augmented hơn). Nếu muốn chọn tay, đặt `VERSION_OVERRIDE` = số version mong muốn rồi chạy lại cell.

In [ ]:
VERSION_OVERRIDE = None  # vd: 3 — đặt tay nếu muốn override lựa chọn tự động

# Thực tế đã xác nhận qua Bước 1: project công khai này CHỈ có 2 version, và CẢ HAI đều đã
# augmentation (rotate/flip/shear/exposure...), không có bản gốc thuần. "images" của mỗi version
# càng lớn nghĩa là augmentation nhân bản càng nhiều (v6=41366, v3=19430) — vì vậy KHÔNG chọn version
# nhiều ảnh nhất (sẽ kéo theo nhiều bản sao augmented hơn, tăng rủi ro leakage giữa train/val/test),
# mà chọn version ÍT ảnh nhất trong 2 lựa chọn hiện có làm mặc định.
with_images = [v for v in version_info if v["images"]]
if VERSION_OVERRIDE is not None:
    CHOSEN_VERSION = VERSION_OVERRIDE
elif with_images:
    CHOSEN_VERSION = min(with_images, key=lambda v: v["images"])["version"]
else:
    CHOSEN_VERSION = max(v["version"] for v in version_info if v["version"] is not None)

print("CHOSEN_VERSION =", CHOSEN_VERSION)
print(
    "[LƯU Ý] Không version nào của project này là bản gốc chưa augment (xem 'augmentation' ở Bước 1). "
    "Đã chọn version ÍT ảnh nhất để giảm số bản sao augmented, nhưng vẫn cần Bước 5 của notebook build "
    "(01_build_tomato_leaf_disease_v1.ipynb) nhóm các ảnh augmented cùng gốc vào chung 1 split — "
    "nếu không, dedup bằng pHash một mình có thể bỏ sót ảnh xoay/lật/nghiêng."
)

LEAF_ROBOFLOW_DIR = RAW / "leaf_roboflow"

already_has_data = LEAF_ROBOFLOW_DIR.exists() and any(LEAF_ROBOFLOW_DIR.rglob("*.jpg"))
if already_has_data:
    print(f"[skip] {LEAF_ROBOFLOW_DIR} đã có dữ liệu, không tải lại.")
else:
    # Không tự mkdir LEAF_ROBOFLOW_DIR trước khi gọi .download(): một số phiên bản Roboflow SDK
    # coi thư mục đích đã tồn tại (dù rỗng) là "đã tải rồi" và bỏ qua download/giải nén thật sự.
    version_obj = project.version(CHOSEN_VERSION)
    dataset = version_obj.download("yolov8", location=str(LEAF_ROBOFLOW_DIR), overwrite=True)
    print("SDK báo đã tải về:", getattr(dataset, "location", LEAF_ROBOFLOW_DIR))

print("\nNội dung raw/leaf_roboflow/:")
found_files = sorted(LEAF_ROBOFLOW_DIR.rglob("*"))[:20] if LEAF_ROBOFLOW_DIR.exists() else []
for p in found_files:
    print(" -", p.relative_to(LEAF_ROBOFLOW_DIR))

if not any(LEAF_ROBOFLOW_DIR.rglob("*.jpg")) if LEAF_ROBOFLOW_DIR.exists() else True:
    print("\nKhông thấy ảnh .jpg nào trong raw/leaf_roboflow/ sau khi tải -> kiểm tra toàn bộ /kaggle/working "
          "xem SDK có ghi ra chỗ khác không:")
    for p in sorted(Path("/kaggle/working").rglob("*.jpg"))[:10]:
        print(" -", p)
    raise RuntimeError(
        "Tải xong nhưng raw/leaf_roboflow/ rỗng. Có thể do thư mục đích đã tồn tại trước khi gọi "
        ".download() khiến SDK bỏ qua giải nén. Thử: xóa hẳn thư mục raw/leaf_roboflow (nếu có) rồi "
        "chạy lại cell này, hoặc kiểm tra danh sách .jpg in phía trên để tìm đúng vị trí thật."
    )

### Bước 3 — Tải Zenodo (nguồn bổ sung, chỉ để audit)
Link đã xác minh trực tiếp trên trang record Zenodo (không đoán tên file): `Tomato Leaf Disease Detection.v1i.yolov8.zip` (~1.5 GB, CC BY 4.0). File này đã được Zenodo export sẵn dạng YOLOv8 kèm resize 640×640 + augmentation — cần lưu ý khi audit vì không phải ảnh gốc thuần.

In [ ]:
ZENODO_URL = (
    "https://zenodo.org/records/20004230/files/"
    "Tomato%20Leaf%20Disease%20Detection.v1i.yolov8.zip?download=1"
)
zenodo_archive = RAW / "_archives" / "leaf_zenodo.zip"
zenodo_dir = RAW / "leaf_zenodo"

# File ~1.5 GB -> 180s (mặc định của download_file) không đủ, timeout=180 từng làm curl bị kill
# giữa chừng dù đang tải bình thường. Tăng timeout lên 1800s (30 phút) cho riêng lần tải này.
download_file(ZENODO_URL, zenodo_archive, timeout=1800)
if not zenodo_dir.exists() or not any(zenodo_dir.iterdir()):
    extract_archive(zenodo_archive, zenodo_dir)
else:
    print(f"[skip] {zenodo_dir} đã giải nén rồi.")

print("\nNội dung raw/leaf_zenodo/:")
for p in sorted(zenodo_dir.rglob("*"))[:20]:
    print(" -", p.relative_to(zenodo_dir))

### Bước 4 — Ghi `sources.csv` (nối thêm) và kiểm kê sơ bộ

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
LABEL_EXTS = {".txt"}

SOURCES = [
    dict(
        dataset_id="leaf_roboflow",
        name="Tomato Leaf Disease (Roboflow)",
        url=f"https://universe.roboflow.com/{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}",
        download_url=f"Roboflow API - version {CHOSEN_VERSION}",
        version=str(CHOSEN_VERSION),
        license="CC BY 4.0",
        task="detection",
        extracted=LEAF_ROBOFLOW_DIR,
        notes="Nguon chinh cho tomato_leaf_disease_v1; class Healthy se thanh negative sample.",
    ),
    dict(
        dataset_id="leaf_zenodo",
        name="Tomato Leaf Disease Detection (Zenodo)",
        url="https://zenodo.org/records/20004230",
        download_url=ZENODO_URL,
        version="v1",
        license="CC BY 4.0",
        task="detection",
        extracted=zenodo_dir,
        notes="CHUA gop vao tomato_leaf_disease_v1 - chi audit (da export san 640x640 + augmentation).",
    ),
]

rows, inventory_rows = [], []
for src in SOURCES:
    extracted = Path(src["extracted"])
    exists = extracted.exists() and any(extracted.iterdir())
    n_images = n_labels = 0
    if exists:
        for p in extracted.rglob("*"):
            if p.suffix.lower() in IMAGE_EXTS:
                n_images += 1
            elif p.suffix.lower() in LABEL_EXTS:
                n_labels += 1
    rows.append({
        "dataset_id": src["dataset_id"], "name": src["name"], "url": src["url"],
        "download_url": src["download_url"], "version": src["version"],
        "download_date": DOWNLOAD_DATE, "license": src["license"], "task": src["task"],
        "sha256": "", "extracted_path": str(extracted) if exists else "", "notes": src["notes"],
    })
    inventory_rows.append({
        "dataset_id": src["dataset_id"], "extracted": exists,
        "n_image_files": n_images, "n_label_like_files": n_labels,
    })

sources_df = pd.DataFrame(rows)
inventory_df = pd.DataFrame(inventory_rows)

sources_df.to_csv(MANIFESTS / "sources_leaf.csv", index=False)
inventory_df.to_csv(MANIFESTS / "dataset_inventory_leaf.csv", index=False)

print(sources_df[["dataset_id", "license", "extracted_path"]].to_string(index=False))
print()
print(inventory_df.to_string(index=False))

## Kết quả
- `raw/leaf_roboflow/` — dữ liệu chính, đã export dạng YOLOv8 (thường có sẵn `train/valid/test` + `data.yaml`).
- `raw/leaf_zenodo/` — dữ liệu bổ sung, **chưa dùng để train**, chỉ để audit/so sánh trùng lặp.
- `manifests/sources_leaf.csv`, `manifests/dataset_inventory_leaf.csv`.

## Trước khi Save Version
- Kiểm tra `dataset_inventory_leaf.csv`: `n_image_files` của `leaf_roboflow` phải > 0 và gần với số ảnh version đã chọn ở Bước 1 (nếu lệch nhiều, version có thể tải thiếu).
- Nếu Roboflow trả lỗi 401/403 ở Bước 1, kiểm tra lại secret `ROBOFLOW_API_KEY` đã attach đúng chưa.

## Bước tiếp theo
Notebook **`01_build_tomato_leaf_disease_v1.ipynb`** sẽ:
1. Attach output của notebook này làm input.
2. Khám phá cấu trúc + liệt kê tên class gốc thật có trong `leaf_roboflow` (đối chiếu với 11 class công khai đã biết).
3. Remap 4 bệnh mục tiêu, chuyển `Healthy` thành negative sample (ảnh giữ, label rỗng), loại các box ngoài phạm vi MVP.
4. Phát hiện ảnh trùng/near-duplicate, chia train/val/test chống leakage.
5. Audit chéo với `leaf_zenodo` (số ảnh, class, mức trùng lặp) để có cơ sở quyết định gộp ở phiên bản sau — **không tự động gộp**.
6. Sinh `tomato_leaf_disease_v1` hoàn chỉnh + manifest, sẵn sàng Save Version thành Kaggle Dataset.